# InSituCNV tutorial: 1105_BL Xenium breast cancer sample

This notebook shows the reusable package workflow on the prepared 1105_BL dataset:

`/home/augusta/storage3/augusta/insituCNV/InSituCNV/Breast_cancer_Xenium5K/01_InSituCNV/data/1105_BL.h5ad`

It follows the manuscript notebook logic, but uses the public `insitucnv` API:
normalize raw counts, smooth over neighbors, normalize and log-transform, add genomic positions, run `infercnvpy`, cluster at multiple resolutions, plot heatmaps and spatial CNV clusters, and export tables.

## 1. Setup

Run this notebook from the package environment:

```bash
conda activate insitucnv_env
cd /home/augusta/storage3/augusta/insituCNV/insituCNV_package/InSituCNV
pip install -e .
jupyter lab
```

In [ ]:
from pathlib import Path

import scanpy as sc
import pandas as pd
import insitucnv as icv

sc.settings.verbosity = 2
sc.set_figure_params(figsize=(6, 6), dpi=100)

DATA_PATH = Path("/home/augusta/storage3/augusta/insituCNV/InSituCNV/Breast_cancer_Xenium5K/01_InSituCNV/data/1105_BL.h5ad")
OUTPUT_DIR = Path("tutorial_outputs/1105_BL")
PLOTS_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH.exists(), DATA_PATH

## 2. Load the prepared AnnData

This file already has the pieces needed by the tutorial:

- raw counts in `adata.layers["raw_counts"]`
- cell type labels in `adata.obs["cell_type_oct25"]`
- spatial coordinates in `adata.obsm["spatial"]`
- a precomputed neighbor graph used for smoothing

In [ ]:
adata = sc.read_h5ad(DATA_PATH)
adata

In [ ]:
print("layers:", list(adata.layers.keys()))
print("obsm:", list(adata.obsm.keys()))
print("obs columns:", list(adata.obs.columns))
print("cell types:", sorted(adata.obs["cell_type_oct25"].astype(str).unique()))

In [ ]:
sc.pl.spatial(
    adata,
    color="cell_type_oct25",
    spot_size=15,
    title="1105_BL cell type annotations",
)

## 3. Define inferCNV reference cells

The reference categories are the non-epithelial cell types used in the manuscript workflow. Adjust this list for other datasets if the annotation names differ.

In [ ]:
reference_key = "cell_type_oct25"
reference_categories = [
    "T_cells",
    "B_cells",
    "Myeloid",
    "Plasma",
    "Fibroblast",
    "Endothelial",
    "Adipocytes",
    "PVLs",
]

present_reference_categories = [cat for cat in reference_categories if cat in set(adata.obs[reference_key].astype(str))]
present_reference_categories

## 4. Normalize, smooth, log-normalize, and add genomic positions

This is the package version of the repeated notebook block:

1. restore `raw_counts` into `X`
2. normalize total counts into `adata.layers["norm"]`
3. smooth normalized counts over 100 neighbors into `adata.layers["M"]`
4. normalize and log-transform the smoothed matrix into `adata.layers["log_norm"]`
5. subset to genes with genomic positions and add `chromosome`, `start`, and `end`

In [ ]:
adata = icv.tl.prepare_cnv_input(
    adata,
    raw_layer="raw_counts",
    target_sum=1e4,
    smoothing_neighbors=100,
    normalized_layer="norm",
    smoothed_layer="M",
    log_layer="log_norm",
)

adata

In [ ]:
adata.var[["chromosome", "start", "end"]].head()

## 5. Run inferCNV

The defaults below match the manuscript workflow for this sample: `window_size=60`, `step=10`, `lfc_clip=4`, and gene-level CNV values enabled.

In [ ]:
icv.tl.run_infercnv(
    adata,
    reference_key=reference_key,
    reference_categories=present_reference_categories,
    input_layer="log_norm",
    window_size=60,
    step=10,
    lfc_clip=4,
    chunksize=1000,
    calculate_gene_values=True,
)

adata

## 6. Compute CNV neighbors and cluster at multiple resolutions

Testing several resolutions lets you compare how granular the CNV clone structure should be. The keys created here are `cnv_leiden_res0.1`, `cnv_leiden_res0.2`, and `cnv_leiden_res0.3`.

In [ ]:
icv.tl.compute_cnv_neighbors(adata)

resolutions = [0.1, 0.2, 0.3]
cluster_keys = icv.tl.cluster_cnv_resolutions(
    adata,
    resolutions=resolutions,
    dendrogram=True,
)

cluster_keys

In [ ]:
for key in cluster_keys:
    display(pd.crosstab(adata.obs[key], adata.obs[reference_key]))

## 7. Plot CNV heatmaps and spatial CNV clusters

These plots are saved in `tutorial_outputs/1105_BL/plots` and also displayed in the notebook.

In [ ]:
for key in cluster_keys:
    print(key)
    icv.pl.plot_chromosome_heatmap(
        adata,
        groupby=key,
        output_path=PLOTS_DIR / f"{key}_heatmap.png",
        dendrogram=True,
        vmin=-0.4,
        vmax=0.4,
        show=False,
    )
    icv.pl.plot_spatial(
        adata,
        color=key,
        output_path=PLOTS_DIR / f"{key}_spatial.png",
        point_size=15,
        title=f"1105_BL {key}",
        show=True,
    )

## 8. Choose a primary resolution and annotate CNV status

For the manuscript sample-by-sample workflow, 1105_BL was inspected at resolution 0.1. Here the package automatically marks the lowest CNV-burden cluster as normal-like and all other clusters as tumor-like. You can also pass explicit `tumor_clusters=[...]` if you want manual control.

In [ ]:
primary_key = "cnv_leiden_res0.1"

icv.tl.calculate_cnv_burden(adata)
icv.tl.assign_cnv_status(
    adata,
    cluster_key=primary_key,
    output_key="cnv_status",
)

pd.crosstab(adata.obs[primary_key], adata.obs["cnv_status"])

In [ ]:
icv.pl.plot_spatial(
    adata,
    color="cnv_status",
    output_path=PLOTS_DIR / "cnv_status_spatial.png",
    point_size=15,
    title="1105_BL CNV status",
    show=True,
)

## 9. Optional: epithelial-only CNV clustering

This mirrors the exploratory notebook pattern where epithelial cells are clustered by CNV profile and all other cells are labeled as `non-epi`.

In [ ]:
epi_cluster_keys = icv.tl.cluster_cnv_resolutions(
    adata,
    resolutions=[0.1, 0.2, 0.3],
    key_prefix="epi_cnv_leiden_res",
    subset_key=reference_key,
    subset_values=["Epithelial"],
    subset_label_prefix="epi_",
    outside_label="non-epi",
    dendrogram=True,
)

epi_cluster_keys

In [ ]:
for key in epi_cluster_keys:
    icv.pl.plot_spatial(
        adata,
        color=key,
        output_path=PLOTS_DIR / f"{key}_spatial.png",
        point_size=15,
        title=f"1105_BL {key}",
        show=True,
    )

## 10. Export results

The gene-level table uses `adata.layers["gene_values_cnv"]`, which was created by `infercnvpy` because `calculate_gene_values=True`.

In [ ]:
mean_cnv = icv.tl.export_mean_cnv_per_gene(
    adata,
    OUTPUT_DIR / "1105_BL_tumor_mean_cnv_per_gene.tsv",
    layer="gene_values_cnv",
    obs_key="cnv_status",
    obs_values=("tumor",),
)

mean_cnv.head()

In [ ]:
icv.tl.export_cell_groups(
    adata,
    OUTPUT_DIR / "1105_BL_cnv_status_xenium_cell_groups.csv",
    group_key="cnv_status",
)

adata.write(OUTPUT_DIR / "1105_BL_CNV_tutorial.h5ad", compression="gzip")
OUTPUT_DIR

## 11. Optional one-command package workflow

After you are comfortable with the step-by-step version above, the same package also has a high-level runner. This cell is left commented so it does not rerun the full analysis by accident.

In [ ]:
# result = icv.run_insitucnv(
#     adata=sc.read_h5ad(DATA_PATH),
#     output_dir="tutorial_outputs/1105_BL_one_command",
#     reference_key="cell_type_oct25",
#     reference_categories=present_reference_categories,
#     raw_layer="raw_counts",
#     smoothing_neighbors=100,
#     window_size=60,
#     step=10,
#     lfc_clip=4,
#     cluster_resolutions=[0.1, 0.2, 0.3],
#     primary_resolution=0.1,
# )